# Description Generation

This notebook performs description generation using a local LLM (Qwen 3
via Ollama).

Objectives:
1. Load the enriched homestay dataset.
2. Generate a factual description for each homestay using Qwen 3.
3. Save descriptions back into the same dataset file.


In [1]:
# ==========================================
# IMPORT LIBRARIES
# ==========================================

import pandas as pd
from ollama import chat
from tqdm import tqdm
import os
import time

In [2]:
# ==========================================
# CONFIGURATION
# ==========================================

INPUT_OUTPUT_FILE = "../data/processed/homestays_enriched.csv"

MODEL_NAME = "qwen3:1.7b"

CHECKPOINT_EVERY_N_ROWS = 10

FINAL_OUTPUT_COLUMN = "description"

CHECKPOINT_DIR = "../data/processed/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

CHECKPOINT_FILE = f"{CHECKPOINT_DIR}/description_generation_checkpoint.csv"

In [3]:
# ==========================================
# LOAD ENRICHED DATASET
# ==========================================

df = pd.read_csv(INPUT_OUTPUT_FILE)

print(f"Total Records: {len(df)}")
df.head()

Total Records: 813


,homestay_id,homestay_name,owner_name,category,district,block,village,owner_email,owner_mobile,google_name,...,lolegaon_proximity,price,wifi,parking,breakfast,mountain_view,room_service,bonfire_barbeque,pickup_dropoff_service,description
0,1,Revere Homestay,Mr. Riwaj Pradhan,Silver,KALIMPONG,Municipality,"8th Mile, Kalimpong",riwajpradhan10@gmail.com,9800686780,Revere Homsetay,...,Far,2126,1,1,1,1,0,0,0,"Revere Homestay, in 8th Mile, Kalimpong, Munic..."
1,2,Mansarover Homestay,Miss Tina Mani Gurung,Gold,KALIMPONG,Municipality,"Chandralok, Kalimpong",santabgurung53@gmail.com,9932234895,Mansarover Homestay / Flora & Transport,...,Far,3959,1,1,1,0,1,1,0,"Mansarover Homestay is located in Chandralok, ..."
2,3,BETHANY HOMESTAY,ANUPAMA TAMANG,Silver,KALIMPONG,Kalimpong 1,Dr.GRAHAMS HOME BLOCK B,wangchuck20199@gmial.com,8348993048,Bethany Homestay Kalimpong,...,Moderately Close,1860,0,0,1,1,0,0,0,"Nestled in the heart of Kalimpong 1, Dr.GRAHAM..."
3,5,BAJARANGI HOMESTAY,KAMAL KUMAR SHARMA,Silver,KALIMPONG,Kalimpong 1,SINGI SAMALBONG KALIMPONG,bajrangihomestay@gmail.com,8670450557,Bajrangi Homestay,...,Moderately Close,2130,0,1,1,1,1,0,0,"BAJARANGI HOMESTAY, located in SINGI SAMALBONG..."
4,7,RELLY VIEW HOMESTAY,SOMA SUNDAS,Silver,KALIMPONG,Kalimpong 1,DR.GRAHAMS HOMES BLOCK B \r\nKALIMPONG,somasundas55@gmail.com,9932095235,Kalimpong View Home Stay,...,Moderately Close,2095,1,0,1,0,0,1,1,"Relly View Homestay is located in Kalimpong 1,..."


In [4]:
# ==========================================
# WRITING STYLE SELECTION
# ==========================================

def get_style(row):

    # Amenity-focused if the homestay has a mountain view
    if row.get("mountain_view", 0) == 1:
        return "amenity-focused"

    # Accessibility-focused if very close to town
    if row.get("town_proximity", "") == "Very Close":
        return "accessibility-focused"

    # Rating-focused if highly rated
    if float(row.get("rating", 0) or 0) >= 4.5:
        return "rating-focused"

    # Default style
    return "location-focused"

In [5]:
# ==========================================
# PROMPT CONSTRUCTION
# ==========================================

AMENITY_COLUMNS = [
    "wifi", "parking", "breakfast", "mountain_view",
    "room_service", "bonfire_barbeque", "pickup_dropoff_service",
]

def create_prompt(row):

    style = get_style(row)

    amenities = [
        col.replace("_", " ").title()
        for col in AMENITY_COLUMNS
        if row.get(col, 0) == 1
    ]
    amenities_text = ", ".join(amenities) if amenities else "None listed"

    prompt = f"""
    Generate a factual homestay description.

    Writing Style: {style}

    Rules:
    - Use ONLY the supplied information.
    - Do NOT invent facts.
    - Do NOT mention facilities that are not listed.
    - Do NOT mention mountain views unless available.
    - You MUST explicitly state the category (Gold or Silver) somewhere in the description -- this is required, not optional.
    - Write AT LEAST 55 words and no more than 80 words. Reach this length by including MORE SPECIFIC real details already supplied (exact proximity to each of Deolo/Durpin/town, the specific village and block names, the exact rating and review count, every listed amenity) -- NOT by adding generic filler phrases like "a wonderful experience" or "nestled in the hills" or "a memorable stay." Every sentence must convey a specific fact from the data above.
    - Do not include word counts, notes, brackets, or explanations.
    - Return ONLY the description.

    Style Instructions:

    Location-focused:
    Start by describing where the homestay is situated.

    Amenity-focused:
    Highlight the facilities first.

    Accessibility-focused:
    Focus on proximity to town, Deolo, and Durpin.

    Rating-focused:
    Naturally mention ratings and reviews.

    Data:

    Name: {row.get('homestay_name', '')}
    Village: {row.get('village', '')}
    Block: {row.get('block', '')}
    Category: {row.get('category', '')}

    Rating: {row.get('rating', '')}
    Review Count: {row.get('review_count', '')}

    Distance to Town:
    {row.get('town_proximity', '')}

    Distance to Deolo:
    {row.get('deolo_proximity', '')}

    Distance to Durpin:
    {row.get('durpin_proximity', '')}

    Amenities:
    {amenities_text}
    """

    return prompt

In [6]:
# ==========================================
# THINKING TRACE CLEANUP
# ==========================================

def strip_thinking(text):
    if "</think>" in text:
        text = text.split("</think>")[-1]
    return text.strip()

In [7]:
# ==========================================
# GENERATION LOOP
# ==========================================

def generate_descriptions(df, checkpoint_path=CHECKPOINT_FILE, checkpoint_every=CHECKPOINT_EVERY_N_ROWS):

    # Resume from checkpoint if one exists
    if os.path.exists(checkpoint_path):
        progress_df = pd.read_csv(checkpoint_path)
        descriptions = progress_df[FINAL_OUTPUT_COLUMN].tolist()
        print(f"Resuming: {len(descriptions)} rows already done.")
    else:
        descriptions = []
        print("No checkpoint -- starting from row 0.")

    start_row = len(descriptions)
    end_row = len(df)

    if start_row >= end_row:
        print("Already complete.")
        return descriptions

    start_time = time.time()

    for idx in tqdm(range(start_row, end_row), initial=start_row, total=end_row):

        row = df.iloc[idx]
        prompt = create_prompt(row)

        try:
            # num_ctx capped at 4096 -- prompt+response fit well under this;
            # Ollama was defaulting to 262144 and running mostly on CPU as a result
            response = chat(
                model=MODEL_NAME,
                messages=[{"role": "user", "content": prompt}],
                options={"num_ctx": 4096},
            )
            description = strip_thinking(response["message"]["content"])

        except Exception as e:
            # Save progress before re-raising so nothing generated so far is lost
            print(f"\nError at row {idx}: {e}")
            _save_checkpoint(df, descriptions, checkpoint_path)
            print("Checkpoint saved -- re-run to resume.")
            raise

        descriptions.append(description)

        # Periodic checkpoint + remaining-time estimate
        if (idx + 1) % checkpoint_every == 0:
            _save_checkpoint(df, descriptions, checkpoint_path)
            elapsed = time.time() - start_time
            avg_time = elapsed / (len(descriptions) - start_row)
            remaining = avg_time * (end_row - len(descriptions))
            print(f"\nRow {idx + 1}/{end_row} | avg {avg_time:.1f}s/row | ~{remaining/60:.1f} min left")

    _save_checkpoint(df, descriptions, checkpoint_path)
    print(f"\nDone: {len(descriptions)} descriptions.")
    return descriptions


def _save_checkpoint(df, descriptions, checkpoint_path):
    # Saves partial progress to disk so a run can resume after interruption
    temp_df = df.iloc[:len(descriptions)].copy()
    temp_df[FINAL_OUTPUT_COLUMN] = descriptions
    temp_df.to_csv(checkpoint_path, index=False)

In [8]:
# ==========================================
# RUN GENERATION
# ==========================================

descriptions = generate_descriptions(df)
df["description"] = descriptions

No checkpoint -- starting from row 0.


  0%|          | 0/813 [00:00<?, ?it/s]

  1%|          | 10/813 [01:37<1:57:47,  8.80s/it]


Row 10/813 | avg 9.7s/row | ~130.1 min left


  2%|▏         | 20/813 [02:45<1:43:56,  7.86s/it]

...


Row 800/813 | avg 8.1s/row | ~1.8 min left


100%|█████████▉| 810/813 [1:48:54<00:21,  7.13s/it]


Row 810/813 | avg 8.1s/row | ~0.4 min left


100%|██████████| 813/813 [1:49:11<00:00,  8.06s/it]


Done: 813 descriptions.


In [9]:
# ==========================================
# VALIDATION CHECKS
# ==========================================

# Check for any empty/failed descriptions
empty_count = (df["description"].str.strip() == "").sum()
print(f"Empty descriptions: {empty_count}")

# Word count distribution (target: 55-80 words)
word_counts = df["description"].str.split().str.len()
print("\nWord count distribution:")
print(word_counts.describe())

# Print a few samples for a manual quality check
print("\nSamples:")
for i in df.sample(min(3, len(df)), random_state=42).index:
    print(f"\n[{df.loc[i, 'homestay_name']}] ({get_style(df.loc[i])})")
    print(df.loc[i, "description"])

Empty descriptions: 1

Word count distribution:
count    813.000000
mean      53.185732
std       10.979883
min        0.000000
25%       46.000000
50%       53.000000
75%       61.000000
max       89.000000
Name: description, dtype: float64

Samples:

[Divine Homestay] (amenity-focused)
Divine Homestay, located in Lava Bazar, PO Lava, Block LAVA, offers a Silver-rated stay with a 4.9 rating and 7 reviews. Situated far from town, Deolo, and Durpin, the homestay provides Wifi, Parking, Breakfast, and Pickup Dropoff Service. Each guest receives a comfortable, well-maintained space with modern facilities, ensuring a pleasant and convenient experience. The homestay includes a Mountain View, making it a unique and immersive stay.

[Toshi Homestay] (amenity-focused)
Toshi Homestay, located in 21st Mile, Dokyong, Pedong, Silver category, is very close to town, moderately close to Deolo, and very close to Durpin. The homestay offers Wifi, Parking, Breakfast, and Mountain View, with Room Servic

In [10]:
# ==========================================
# SAVE ENRICHED DATASET WITH DESCRIPTIONS
# ==========================================

df.to_csv(INPUT_OUTPUT_FILE, index=False)

print("Description Generation Completed.")
print(f"Saved to: {INPUT_OUTPUT_FILE}")

Description Generation Completed.
Saved to: ../data/processed/homestays_enriched.csv
